
# Distorted Visual Sequence Pattern Recognition
**Name:** Ishant Bharti  
**Institute:** IIT Roorkee  

---

## The Problem
This challenge presents a grayscale sequence dataset featuring severe, real-world visual degradations designed to break standard heuristics. The grayscale sequence dataset features **non-linear spatial warping**, **symbol overlap**, **blur**, and **targeted occlusion patches**. To solve this, the model cannot simply look at local pixel patterns; it must build a contextual understanding of the sequence to infer identity when characters are completely occluded.



## The Lightweight CRNN + CTC Pivot
Our initial approach explored massive Vision-Language transformers (like TrOCR). While accurate, training 30M+ parameter autoregressive models for ensembling was heavily bottlenecked by hardware compute and iteration speed.

To overcome these limitations and achieve lightning-fast iterations without sacrificing accuracy, we pivoted to a **Highly Optimized Convolutional Recurrent Neural Network (CRNN)** combined with Connectionist Temporal Classification (CTC) Loss:
- **CNN Extractor:** A lightweight 3-layer Convolutional stack dynamically reduces the spatial dimensions of the $50\times 200$ images while extracting high-level character features.
- **Bi-LSTM Sequence Modeler:** Bidirectional LSTMs interpret the sequential feature map, providing the forward and backward context necessary to identify characters even when partially obscured.
- **CTC Loss Advantage:** CTC elegantly handles the unaligned nature of the characters without requiring strict bounding boxes, and it is significantly faster to compute and decode than autoregressive generation.



## The Secret Weapon: Augmentations & Ensembling
A tiny CRNN (only ~600k parameters) is prone to overfitting. To bridge the gap and achieve the extremely strict **0.0008 CER** threshold, we supercharged the architecture with two critical upgrades:
1. **Aggressive `torchvision.transforms.v2` Augmentations:** We artificially inject Color Jitter and Random Rotations during training. This forces the lightweight CRNN to generalize rather than memorizing specific pixel patterns of the occlusions.
2. **Logit-Averaging Multi-Seed Ensembling:** We train 3 distinct instances of the model on different random seeds. During inference, instead of majority voting the final strings, we strictly **average the raw CTC probability logits** at each timestep across all 3 models *before* applying the greedy collapse. This produces a massive boost in accuracy by smoothing out model-specific hallucinations.


In [ ]:

import os
import sys
import random
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.text import CharErrorRate
from torchvision.transforms import v2

# Ensure inline plotting is enabled
%matplotlib inline

# Fix working directory if run from inside submissions folder
if os.path.basename(os.getcwd()) == 'submissions':
    os.chdir('..')
sys.path.append(os.getcwd())

torch.set_float32_matmul_precision('medium')
import warnings
warnings.filterwarnings('ignore')


### 1. Tokenizer Configuration

In [ ]:

class CharTokenizer:
    def __init__(self):
        chars = ['+', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'r']
        
        # CTC Blank is index 0
        self.pad_token = '<PAD>'
        self.sos_token = '<SOS>'
        self.eos_token = '<EOS>'
        self.unk_token = '<UNK>'
        
        self.special_tokens = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]
        self.vocab = self.special_tokens + chars
        
        self.char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        self.id_to_char = {idx: char for idx, char in enumerate(self.vocab)}
        
        self.pad_id = self.char_to_id[self.pad_token]
        self.sos_id = self.char_to_id[self.sos_token]
        self.eos_id = self.char_to_id[self.eos_token]
        self.unk_id = self.char_to_id[self.unk_token]


### 2. Dataset & Data Augmentation

In [ ]:

class OCRDataset(Dataset):
    def __init__(self, csv_file=None, image_dir=None, file_col='image', label_col='text', is_test=False, transform=None):
        self.image_dir = image_dir
        self.is_test = is_test
        self.tokenizer = CharTokenizer()
        
        if transform is None:
            if not self.is_test:
                self.transform = v2.Compose([
                    v2.Grayscale(num_output_channels=1),
                    v2.Resize((50, 200)),
                    v2.RandomRotation(5),
                    v2.ColorJitter(brightness=0.2, contrast=0.2),
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),
                ])
            else:
                self.transform = v2.Compose([
                    v2.Grayscale(num_output_channels=1),
                    v2.Resize((50, 200)),
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),
                ])
        else:
            self.transform = transform

        if not self.is_test:
            df = pd.read_csv(csv_file, usecols=[file_col, label_col])
            df = df.dropna(subset=[label_col])
            self.data = list(zip(df[file_col].astype(str), df[label_col].astype(str)))
        else:
            self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.image_files) if self.is_test else len(self.data)

    def __getitem__(self, idx):
        if self.is_test:
            img_name = self.image_files[idx]
            img_path = os.path.join(self.image_dir, img_name)
            image = Image.open(img_path).convert('RGB')
            return img_name, self.transform(image)
        else:
            img_name, text = self.data[idx]
            img_path = os.path.join(self.image_dir, img_name)
            image = Image.open(img_path).convert('RGB')
            
            # For CTC, we only need character IDs
            labels = [self.tokenizer.char_to_id.get(char, self.tokenizer.unk_id) for char in text]
            return self.transform(image), torch.tensor(labels, dtype=torch.long)

def collate_fn(batch):
    if isinstance(batch[0][0], str):
        img_names = [item[0] for item in batch]
        pixel_values = torch.stack([item[1] for item in batch])
        return img_names, pixel_values

    pixel_values = torch.stack([item[0] for item in batch])
    labels = [item[1] for item in batch]
    target_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long)
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)
    return pixel_values, labels_padded, target_lengths


### 3. PyTorch CRNN Architecture

In [ ]:

class CRNN(nn.Module):
    def __init__(self, vocab_size):
        super(CRNN, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2), # -> [B, 32, 25, 100]
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2), # -> [B, 64, 12, 50]
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2)  # -> [B, 128, 6, 25]
        )
        
        self.dense1 = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        
        self.lstm1 = nn.LSTM(128, 128, bidirectional=True, batch_first=True)
        self.dropout1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(256, 64, bidirectional=True, batch_first=True)
        self.dropout2 = nn.Dropout(0.3)
        
        self.classifier = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, w, c * h)
        
        x = self.dense1(x)
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        
        logits = self.classifier(x)
        return logits


### 4. Lightning Model Wrapper & CTC Loss

In [ ]:

class LightningOCR(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.tokenizer = CharTokenizer()
        self.vocab_size = len(self.tokenizer.char_to_id)
        
        self.model = CRNN(vocab_size=self.vocab_size)
        self.criterion = nn.CTCLoss(blank=0, zero_infinity=True) # Blank is <PAD>
        self.cer_metric = CharErrorRate()

    def forward(self, pixel_values):
        return self.model(pixel_values)

    def training_step(self, batch, batch_idx):
        pixel_values, labels, target_lengths = batch
        logits = self(pixel_values)
        
        log_probs = nn.functional.log_softmax(logits, dim=2).permute(1, 0, 2)
        input_lengths = torch.full((pixel_values.size(0),), logits.size(1), dtype=torch.long, device=self.device)
        
        loss = self.criterion(log_probs, labels, input_lengths, target_lengths)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        pixel_values, labels, target_lengths = batch
        logits = self(pixel_values)
        
        log_probs = nn.functional.log_softmax(logits, dim=2).permute(1, 0, 2)
        input_lengths = torch.full((pixel_values.size(0),), logits.size(1), dtype=torch.long, device=self.device)
        
        val_loss = self.criterion(log_probs, labels, input_lengths, target_lengths)
        self.log('val_loss', val_loss, prog_bar=True)
        
        preds = torch.argmax(logits, dim=2)
        
        pred_strings = []
        target_strings = []
        for i in range(pixel_values.size(0)):
            pred_seq = preds[i].tolist()
            decoded_pred = []
            prev_char = -1
            for p in pred_seq:
                if p != prev_char and p != 0:
                    decoded_pred.append(p)
                prev_char = p
            pred_str = "".join([self.tokenizer.id_to_char.get(idx, "") for idx in decoded_pred])
            
            target_seq = labels[i][:target_lengths[i]].tolist()
            target_str = "".join([self.tokenizer.id_to_char.get(idx, "") for idx in target_seq])
            
            pred_strings.append(pred_str)
            target_strings.append(target_str)
            
        self.cer_metric.update(pred_strings, target_strings)
        
    def on_validation_epoch_end(self):
        cer = self.cer_metric.compute()
        self.log('val_cer', cer, prog_bar=True)
        self.cer_metric.reset()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


### 5. Multi-Seed Training Execution
*(Commented out to prevent massive computation runs during notebook review)*

In [ ]:

def train_seed(seed, train_csv, val_csv, raw_img_dir):
    pl.seed_everything(seed, workers=True)
    
    train_dataset = OCRDataset(csv_file=train_csv, image_dir=raw_img_dir, file_col='image', label_col='text')
    val_dataset = OCRDataset(csv_file=val_csv, image_dir=raw_img_dir, file_col='image', label_col='text', is_test=False)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0, collate_fn=collate_fn)

    model = LightningOCR(lr=1e-3)
    
    os.makedirs("checkpoints/ensemble", exist_ok=True)
    checkpoint_callback = ModelCheckpoint(
        dirpath="checkpoints/ensemble/",
        filename=f"crnn-seed-{seed}" + "-{epoch:02d}-{val_cer:.4f}",
        monitor="val_cer",
        mode="min",
        save_top_k=1,
    )

    trainer = pl.Trainer(
        max_epochs=100, # Lightweight model can train for many epochs quickly
        callbacks=[checkpoint_callback],
        gradient_clip_val=1.0,
        precision="16-mixed",
    )

    print(f"\n--- Starting training for SEED {seed} ---")
    trainer.fit(model, train_loader, val_loader)

# def run_full_ensemble_training():
#     raw_csv_path = "data/raw/train-labels.csv"
#     raw_img_dir = "data/raw/train_images"
#     df = pd.read_csv(raw_csv_path).sample(frac=1, random_state=42).reset_index(drop=True)
#     split_idx = int(0.9 * len(df))
#     train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]
#     
#     os.makedirs("data/processed", exist_ok=True)
#     train_df.to_csv("data/processed/train.csv", index=False)
#     val_df.to_csv("data/processed/val.csv", index=False)
#     
#     seeds = [42, 123, 999]
#     for s in seeds:
#         train_seed(s, "data/processed/train.csv", "data/processed/val.csv", raw_img_dir)

# run_full_ensemble_training()


### 6. Ensemble Inference & Prediction

In [ ]:
def ensemble_generate(models, pixel_values):
    """
    Logit-averaging CTC greedy decoding.
    Passes visual features through all models, averages the logits, and collapses.
    """
    ensemble_probs = 0
    for m in models:
        logits = m(pixel_values) # [B, T, C]
        probs = torch.softmax(logits, dim=-1)
        ensemble_probs += probs
        
    ensemble_probs /= len(models)
    
    # Custom fixed-length decoding to force exactly 6 characters
    B, T, C = ensemble_probs.shape
    device = ensemble_probs.device
    preds = torch.zeros((B, 6), dtype=torch.long, device=device)
    
    for i in range(B):
        p = ensemble_probs[i].clone()
        p[:, 0] = 0.0 # Ignore blank token
        max_probs, max_chars = torch.max(p, dim=-1)
        
        blocks = []
        last_char = -1
        for t in range(T):
            c = max_chars[t].item()
            if c != last_char:
                blocks.append((c, max_probs[t].item(), t))
                last_char = c
            else:
                if max_probs[t].item() > blocks[-1][1]:
                    blocks[-1] = (c, max_probs[t].item(), t)
                    
        if len(blocks) >= 6:
            blocks.sort(key=lambda x: x[1], reverse=True)
            best_blocks = blocks[:6]
            best_blocks.sort(key=lambda x: x[2])
            for j in range(6):
                preds[i, j] = best_blocks[j][0]
        else:
            for j in range(len(blocks)):
                preds[i, j] = blocks[j][0]
            for j in range(len(blocks), 6):
                preds[i, j] = blocks[-1][0] if len(blocks) > 0 else 1
                
    return preds



<div style="padding: 20px; background-color: #1a1a1a; border-radius: 10px; margin-top: 20px; border: 1px solid #444; box-shadow: 0px 4px 15px rgba(0,0,0,0.5);">
    <h2 style="color: #ffcc00; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center;">📸 Visualizing the Ensemble's Capability (Dynamic)</h2>
    <p style="font-size: 16px; color: #dcdcdc; line-height: 1.6; text-align: center;">
        Run the code cell below to randomly sample 16 test images, pass them through the <strong>Multi-Seed Ensemble</strong>, and display the predicted text natively in the notebook! <strong>You can run this cell multiple times to see different sets of predictions.</strong>
    </p>
</div>


In [ ]:
import os
import random
import matplotlib.pyplot as plt

def visualize_ensemble_predictions():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ensemble_dir = "checkpoints/ensemble/"
    test_img_dir = "data/raw/test_images"
    
    models = []
    if os.path.exists(ensemble_dir):
        checkpoint_files = [f for f in os.listdir(ensemble_dir) if f.endswith('.ckpt')]
        for ckpt in checkpoint_files:
            model = LightningOCR.load_from_checkpoint(os.path.join(ensemble_dir, ckpt)).to(device).eval()
            models.append(model)
            
    if not models:
        print("No trained models found! Run the training loop first.")
        models.append(LightningOCR().to(device).eval())

    all_images = [f for f in os.listdir(test_img_dir) if f.endswith('.png')]
    selected_images = random.sample(all_images, min(16, len(all_images)))
    
    tokenizer = CharTokenizer()
    
    # We use OCRDataset to get the exact transforms the model expects
    test_dataset = OCRDataset(image_dir=test_img_dir, is_test=True)
    
    fig, axes = plt.subplots(4, 4, figsize=(15, 10))
    axes = axes.flatten()

    print(f"Visualizing logit-averaged predictions from {len(models)} models...")
    with torch.no_grad():
        for i, img_name in enumerate(selected_images):
            # Get the exact index of this image in the dataset
            idx = test_dataset.image_files.index(img_name)
            _, pixel_values = test_dataset[idx]
            
            # Add batch dimension
            pixel_values = pixel_values.unsqueeze(0).to(device)
            
            # Predict
            generated_preds = ensemble_generate(models, pixel_values)
            
            # Decode: we already performed fixed-length extraction
            pred_seq = generated_preds[0].tolist()
            decoded_pred = [p for p in pred_seq if p != 0] # Remove blanks if any
            pred_text = "".join([tokenizer.id_to_char.get(idx, "") for idx in decoded_pred])
            
            # Plot original image (load purely for plotting)
            img_path = os.path.join(test_img_dir, img_name)
            image_disp = Image.open(img_path).convert("RGB")
            
            axes[i].imshow(image_disp)
            axes[i].set_title(f"Pred: {pred_text}", fontsize=14, color='darkgreen', fontweight='bold')
            axes[i].axis('off')

    plt.tight_layout()
    plt.show()

visualize_ensemble_predictions()
